In [0]:
from pyspark.sql import functions as F

catalog, schema = "workspace", "patient_notes"
gold_table = f"{catalog}.{schema}.gold_note_conditions"
eval_table = f"{catalog}.{schema}.eval_runs"
base_path  = f"/Volumes/{catalog}/{schema}/raw_notes"

gold  = spark.table(gold_table)
truth = (spark.read.option("recursiveFileLookup", "true").option("pathGlobFilter", "*.json")
              .json(f"{base_path}/ground_truth"))
print("Gold notes:", gold.count(), "| Answer-key notes:", truth.count())

In [0]:
pred = (gold.select("note_id", F.explode("conditions").alias("p"))
            .select("note_id",
                    F.lower(F.trim("p.condition")).alias("condition"),
                    F.lower(F.trim("p.status")).alias("pred_status")))

true = (truth.join(gold.select("note_id"), "note_id")          # grade only the notes the AI processed
             .select("note_id", F.explode("labels").alias("t"))
             .select("note_id",
                     F.lower("t.condition").alias("condition"),
                     F.col("t.status").alias("true_status")))

both = pred.join(true, ["note_id", "condition"], "full_outer")   # keeps misses AND inventions
display(both.orderBy("note_id"))

In [0]:
n_pred  = pred.count()
n_true  = true.count()
n_found = both.filter("pred_status IS NOT NULL AND true_status IS NOT NULL").count()
n_exact = both.filter("pred_status = true_status").count()

cond_p, cond_r   = n_found / n_pred, n_found / n_true
exact_p, exact_r = n_exact / n_pred, n_exact / n_true

print(f"Predicted pairs: {n_pred} | True pairs: {n_true}")
print(f"Condition only      -> precision {cond_p:.0%}, recall {cond_r:.0%}")
print(f"Condition + status  -> precision {exact_p:.0%}, recall {exact_r:.0%}")

In [0]:
display(both.fillna("missing", subset=["pred_status", "true_status"])
            .groupBy("true_status").pivot("pred_status").count()
            .orderBy("true_status"))

In [0]:
mistakes = (both.filter("pred_status IS NULL OR true_status IS NULL OR pred_status != true_status")
                .join(gold.select("note_id", "assessment"), "note_id"))
display(mistakes.select("note_id", "assessment", "condition", "true_status", "pred_status")
                .orderBy("note_id"))

In [0]:
prompt_chars = 782   # in 05_ai_extraction run print(len(instructions)) and put the exact number here

display(gold.select(((F.length("assessment") + F.lit(prompt_chars)) / 4).alias("in_tokens"),
                    (F.length(F.to_json("conditions")) / 4).alias("out_tokens"))
            .agg(F.round(F.avg("in_tokens")).alias("avg_input_tokens"),
                 F.round(F.avg("out_tokens")).alias("avg_output_tokens")))

In [0]:
run = (spark.createDataFrame(
          [("v2-prompt", "llama-3.3-70b", n_pred, n_true, cond_p, cond_r, exact_p, exact_r)],
          "prompt_version string, model string, n_pred long, n_true long, "
          "cond_precision double, cond_recall double, exact_precision double, exact_recall double")
       .withColumn("run_at", F.current_timestamp()))

run.write.mode("append").saveAsTable(eval_table)
display(spark.table(eval_table))